### Notebook to train the Teacher and students models

In [1]:
import os
from google.colab import drive, userdata
drive.mount('/content/drive')
token = userdata.get('githubAccess')
os.environ['WANDB_API_KEY'] = userdata.get('wandbKey')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:

%cd /content/drive/MyDrive/incremental_learning_on_the_edge/src/core

/content/drive/MyDrive/incremental_learning_on_the_edge/src/core


In [3]:
!git pull

Already up to date.


In [4]:
%pip install -q "transformers>=4.48,<5" "huggingface_hub>=0.34,<1" "datasets<3.0.0"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 152.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 50.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.3/527.3 kB 49.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.6/177.6 kB 21.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.6.1 which is incompatible.
gradio 6.24.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [26]:
#If errors, run this and restart runTime
#!pip install -q numpy==1.26.4 datasets==2.14.7 huggingface_hub==0.17.3 transformers==4.35.0

In [4]:
import sys
import wandb
import torch
import pandas as pd
from dotenv import load_dotenv
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
)
sys.path.append("..")
from core.configuration import load_config
from core.dataloader import DataClass
from core.utils import build_model
from core.train import Trainer
from core.distillation_1 import DistillationTrainer, DistillationTrainerV0, run_distillation_training, run_incremental_step, run_incremental_experiment


In [5]:
import transformers, sys
print(transformers.__version__)
print(transformers.__file__)
print(sys.executable)

4.57.6
/usr/local/lib/python3.13/dist-packages/transformers/__init__.py
/usr/bin/python3


In [29]:
# def run_distillation_training(
#     dataclass, teacher_model, teacher_tokenizer, models_keys, config
# ):
#     for key in models_keys:
#         student_model, student_tokenizer, _ = build_model(config[key]["name"], dataclass.num_labels)
#         dft_pre_dataloader = dataclass.get_dataloader_data(key, student_tokenizer, keep_utt=True)
#         trainer = DistillationTrainer(
#             student_model=student_model,
#             teacher_model=teacher_model,
#             student_name=key,
#             teacher_tokenizer=teacher_tokenizer,
#             student_dataloaders=dft_pre_dataloader,
#             config=config,
#         )
#         trainer.train()

### Running the distillation loop over the V0 in EN and Default hyperparameters. For tuning change config.yaml

In [6]:
config = load_config()
dataclass = DataClass()
print("Dataset Loaded")
teacher_path = "outputs/checkpoints/teacher_model"
teacher_model = AutoModelForSequenceClassification.from_pretrained(teacher_path)
teacher_tokenizer = AutoTokenizer.from_pretrained(config["teacher"]["name"])
model_keys = ["student1"]
run_distillation_training(dataclass, teacher_model, teacher_tokenizer, model_keys, config)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


The repository for qanastek/MASSIVE contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/qanastek/MASSIVE.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N] y


Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

...

Some weights of ModernBertForSequenceClassification were not initialized from the model checkpoint at jhu-clsp/ettin-encoder-68m and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Map:   0%|          | 0/3622 [00:00<?, ? examples/s]

Map:   0%|          | 0/914 [00:00<?, ? examples/s]

Map:   0%|          | 0/657 [00:00<?, ? examples/s]

ValueError: Label-space mismatch: student=20, teacher=15. Distillation requires matching classifier heads.

### Running test of incremental distillation re-training loop in EN and Default hyperparameters. For tuning change config.yaml.


#### We use V0 as teacher and increment the model knowledge by a single intent into the V1 version.

In [ ]:
# config = load_config()
# dataclass = DataClass()

# print("Dataset Loaded")
# print("num_labels:", dataclass.num_labels)                       # must be 15  ==> V0
# print("reserve intents:", dataclass.dataset_totrain["train_set"].unique("intent"))
# new_intent_name = "general_joke"
# run_incremental_step(
#     dataclass=dataclass,
#     new_intent_name=new_intent_name,
#     student_key="student1",
#     config=config,
#     version=1,      # creates version V1 of incremental model
#     K=50,
# )

### Run the experimental run, up to 5 new intents added, so 5 incremental steps

In [6]:
config = load_config()
dataclass = DataClass()
print("Dataset Loaded")
print("num_labels:", dataclass.num_labels)
print("reserve intents:", dataclass.dataset_totrain["train_set"].unique("intent"))
#reserve intents: ['takeaway_order', 'general_joke', 'recommendation_locations', 'play_podcasts', 'transport_traffic']

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Filter:   0%|          | 0/11514 [00:00<?, ? examples/s]

Filter:   0%|          | 0/2974 [00:00<?, ? examples/s]

Filter:   0%|          | 0/2033 [00:00<?, ? examples/s]

Filter:   0%|          | 0/11514 [00:00<?, ? examples/s]

...

Map:   0%|          | 0/122 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/690 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/150 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/122 [00:00<?, ? examples/s]

Dataset Loaded
num_labels: 17
reserve intents: ['takeaway_order', 'general_joke', 'recommendation_locations', 'play_podcasts', 'transport_traffic']


In [11]:
intents_to_add = ["general_joke"]
run_incremental_experiment(intents_to_add, "student1", config, 60, 42)


Filter:   0%|          | 0/11514 [00:00<?, ? examples/s]

Filter:   0%|          | 0/2974 [00:00<?, ? examples/s]

Filter:   0%|          | 0/2033 [00:00<?, ? examples/s]

Filter:   0%|          | 0/11514 [00:00<?, ? examples/s]

Filter:   0%|          | 0/2974 [00:00<?, ? examples/s]

...

testing epoch: 4/20: 100%|██████████| 21/21 [00:00<00:00, 31.03it/s]


Saving checkpoint


testing epoch: 9/20: 100%|██████████| 21/21 [00:00<00:00, 31.96it/s]


Early stopping at epoch 9


epoch,▁▂▃▃▄▅▆▆▇█
eval_accuracy,█▂▅▃▇▄▃▁▃▄
eval_accuracy_en-US,█▂▅▃▇▄▃▁▃▄
eval_loss,▁▆▅▇▂▆▇▇▆█
eval_macro_f1,▆▃▅▄█▆▄▁▄▅
eval_macro_f1_en-US,▆▃▅▄█▆▄▁▄▅
eval_weighted_f1,█▃▆▃█▅▄▁▃▄
eval_weighted_f1_en-US,█▃▆▃█▅▄▁▃▄
lr,▁█▇▆▆▅▄▃▃▂
train_loss,█▃▂▂▂▁▁▁▁▁
+1,...


### Now running 5 incremental steps

In [7]:
intents_to_add = ['takeaway_order', 'general_joke', 'recommendation_locations', 'play_podcasts', 'transport_traffic']
run_incremental_experiment(intents_to_add, "student1", config, 70, 42)

v1: adding intent: takeaway_order


Map:   0%|          | 0/677 [00:00<?, ? examples/s]

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: bruno-santome-antolin (bruno-santome-antolin-city-university-of-london) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


testing epoch: 0/20: 100%|██████████| 22/22 [00:01<00:00, 20.91it/s]


Saving checkpoint


...

testing epoch: 2/20: 100%|██████████| 25/25 [00:00<00:00, 32.66it/s]


Saving checkpoint


testing epoch: 7/20: 100%|██████████| 25/25 [00:00<00:00, 33.12it/s]


Early stopping at epoch 7


epoch,▁▂▃▄▅▆▇█
eval_accuracy,▁▅▇▆█▅█▄
eval_accuracy_en-US,▁▅▇▆█▅█▄
eval_f1_alarm_set,▁▁▁▁▁▁▁▁
eval_f1_audio_volume_down,███▁▁▁█▁
eval_f1_audio_volume_mute,███▁████
eval_f1_audio_volume_up,█▄▄▁▁▁▄▁
eval_f1_datetime_query,██▁▁████
eval_f1_email_addcontact,▁▁▁▁▁▁▁▁
eval_f1_general_joke,▁███████
+21,...


In [ ]:
# !find . -name "*.safetensors" -o -name "pytorch_model.bin" 2>/dev/null

### Evaluating the models

In [6]:
from core.evaluate import  intents_report, old_intent_persistance_table, new_intent_acquisition_table

In [7]:

config = load_config()
dataclass = DataClass()
intents = ['takeaway_order', 'general_joke', 'recommendation_locations', 'play_podcasts', 'transport_traffic']
for name in intents:
  dataclass.admit_intent(name)

rows, metrics = intents_report(dataclass, "student1", config, n_versions=len(intents))
print(old_intent_persistance_table(rows, dataclass.id2intent))
print(new_intent_acquisition_table(rows, dataclass.id2intent))

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

Filter:   0%|          | 0/1064 [00:00<?, ? examples/s]

...

V4: has 19 intents on the test split


Filter:   0%|          | 0/1064 [00:00<?, ? examples/s]

Map:   0%|          | 0/1064 [00:00<?, ? examples/s]

evaluating test set: 100%|██████████| 34/34 [00:01<00:00, 33.65it/s]


V5: has 20 intents on the test split
                         V0        V1        V2        V3        V4        V5
alarm_set          0.975610  0.975610  0.975610  0.987654  0.987654  0.987654
audio_volume_down  1.000000  0.952381  0.952381  0.900000  0.952381  1.000000
audio_volume_mute  0.939394  0.953846  0.937500  0.937500  0.937500  0.939394
audio_volume_up    0.916667  0.833333  0.833333  0.740741  0.769231  0.869565
datetime_query     0.960452  0.960452  0.965909  0.965909  0.960452  0.971429
email_addcontact   0.923077  0.888889  0.923077  0.923077  0.923077  0.923077
lists_createoradd  0.945946  0.897436  0.906667  0.909091  0.933333  0.921053
news_query         0.964427  0.971888  0.967480  0.953975  0.945148  0.935622
play_audiobook     0.805195  0.784810  0.805195  0.805195  0.780488  0.805195
play_game          0.861538  0.848485  0.843750  0.857143  0.848485  0.833333
play_music         0.928962  0.935574  0.935574  0.920000  0.905325  0.913295
play_radio         0.936170